Define a non time-based hierarchical model first

In [ ]:
import jax.numpy as jnp
import jax.random as jr
import numpyro
import numpyro.distributions as dist


def hierarchical_numpyro_model(ys=None, N=None):
    #If ys is provide, make sure N matches ys' length
    if ys is not None:
        if N is not None:
            assert len(ys) == N
        N = len(ys)

    mu_global = numpyro.sample("mu_global", dist.Normal(0.0, 0.5))
    #N is how many mu_i's we sample
    with numpyro.plate("N", N):
        mu_i = numpyro.sample("mu_i", dist.Normal(mu_global, 0.5))
        numpyro.sample("y_i", dist.Normal(mu_i, 0.1), obs=ys)

Generating data and inference

In [ ]:
from numpyro.infer import NUTS, MCMC, Predictive
import seaborn as sns
import matplotlib.pyplot as plt

# Generate data with predictive
predictive = Predictive(hierarchical_numpyro_model, num_samples=1)
synthetic_data = predictive(jr.PRNGKey(0), N=10)

# Infer mu_i and mu_global with NUTS
nuts_kernel = NUTS(hierarchical_numpyro_model)
mcmc = MCMC(nuts_kernel, num_warmup=1000, num_samples=1000)
mcmc.run(jr.PRNGKey(0), ys=synthetic_data["y_i"].squeeze())

# Plot the posterior distribution of mu_i and mu_global
sns.violinplot(data=mcmc.get_samples()["mu_i"], fill=False)
plt.scatter(
    range(10),
    synthetic_data["mu_i"],
    color="black",
    zorder=10,
    marker="x",
    s=100,
    label=r"True μi",
)
sns.despine()
plt.title(r"Posterior Recovery of μi")
plt.xlabel("Individual")
plt.ylabel(r"μi")
plt.legend()
plt.show()

# Plot the posterior of mu_global
sns.histplot(mcmc.get_samples()["mu_global"], bins=20)
plt.axvline(
    synthetic_data["mu_global"],
    color="black",
    linestyle="--",
    lw=5,
    label=r"True μglobal",
)
plt.legend()
plt.title(r"Posterior of μglobal")
plt.xlabel(r"μglobal")
plt.ylabel("Frequency")
sns.despine()
plt.show()

Code defining a dynamic hierarchical/mixed-effect model that follows an OU process

In [ ]:
import dynestyx as dsx
from dynestyx import LTI_continuous, Simulator
from dynestyx.inference.filter_configs import ContinuousTimeKFConfig
from dynestyx.inference.filters import Filter


def hierarchical_ou_model(
    obs_times=None,
    obs_values=None,
    predict_times=None,
    N_trajectories=None,
):
    assert N_trajectories is not None, "N_trajectories must be specified"

    state_dim = 2
    A = jnp.array([[-0.8, 0.25], [-0.15, -0.6]])
    L = 0.20 * jnp.eye(state_dim)
    H = jnp.eye(state_dim)
    R = (0.08**2) * jnp.eye(state_dim)

    mu_global = numpyro.sample(
        "mu_global", dist.MultivariateNormal(jnp.zeros(state_dim), jnp.eye(state_dim) * 0.5**2)
    )
    sigma = numpyro.sample(
        "sigma", dist.HalfNormal(0.4 * jnp.ones(state_dim)).to_event(1)
    )
    mu_0_global = numpyro.sample(
        "mu_0_global", dist.MultivariateNormal(jnp.zeros(state_dim), jnp.eye(state_dim) * 0.7**2)
    )
    sigma_0 = numpyro.sample(
        "sigma_0", dist.HalfNormal(0.5 * jnp.ones(state_dim)).to_event(1)
    )

    with dsx.plate("N", N_trajectories):
        mu_i = numpyro.sample("mu_i", dist.MultivariateNormal(mu_global, sigma * jnp.eye(state_dim)))
        mu_0_i = numpyro.sample(
            "mu_0_i", dist.MultivariateNormal(mu_0_global, sigma_0 * jnp.eye(state_dim))
        )
        b = -jnp.einsum("ij,...j->...i", A, mu_i)


        #Careful that this is not DynamicalModel 
        dynamics = LTI_continuous(
            A=A,
            L=L,
            H=H,
            R=R,
            b=b,
            initial_mean=mu_0_i,
            initial_cov=0.15 * jnp.eye(state_dim),
        )

        return dsx.sample(
            "f",
            dynamics,
            obs_times=obs_times,
            obs_values=obs_values,
            predict_times=predict_times,
        )

Generating real/synthetic data

In [ ]:
N_trajectories = 8
predict_times = jnp.linspace(0.0, 10.0, 100)
true_params = {
    "mu_global": jnp.array([0.8, -0.4]),
    "sigma": jnp.array([0.25, 0.20]),
    "mu_0_global": jnp.array([-0.8, 0.7]),
    "sigma_0": jnp.array([0.25, 0.30]),
}

with Simulator():
    predictive = Predictive(
        hierarchical_ou_model,
        params=true_params,
        num_samples=1,
        exclude_deterministic=False,
    )
    synthetic_data = predictive(
        jr.PRNGKey(1),
        N_trajectories=N_trajectories,
        predict_times=predict_times,
    )

obs_times = predict_times
obs_values = synthetic_data["f_observations"][0, :, 0]

synthetic_data["mu_i"].shape, synthetic_data["mu_0_i"].shape, obs_values.shape

Plotting the data

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for component, ax in enumerate(axes):
    for i in range(N_trajectories):
        color = f"C{i}"
        ax.axhline(
            synthetic_data["mu_i"][0, i, component],
            color=color,
            linestyle="--",
            lw=1,
            alpha=0.8,
        )
        ax.plot(predict_times, obs_values[i, :, component], color=color, alpha=0.8)
        ax.scatter(
            0.0,
            synthetic_data["mu_0_i"][0, i, component],
            color=color,
            edgecolor="black",
            s=45,
            zorder=5,
            label=r"Initial Condition Mean" if i == 0 and component == 0 else None,
        )
    ax.set_ylabel(f"component {component}")
axes[0].legend(loc="best")
axes[-1].set_xlabel("time")
sns.despine()
plt.show()

Inference via Continuous Kalman Filter

In [ ]:
def conditioned_hierarchical_ou_model():
    with Filter(ContinuousTimeKFConfig()):
        return hierarchical_ou_model(
            N_trajectories=N_trajectories,
            obs_times=obs_times,
            obs_values=obs_values,
        )


mcmc = MCMC(NUTS(conditioned_hierarchical_ou_model), num_warmup=100, num_samples=100)
mcmc.run(jr.PRNGKey(2))
posterior = mcmc.get_samples()

print(
    posterior["mu_global"].shape,
    posterior["sigma"].shape,
    posterior["mu_i"].shape,
    posterior["mu_0_global"].shape,
    posterior["sigma_0"].shape,
    posterior["mu_0_i"].shape,
)



Plotting our Inferences

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=False)
for component in range(2):
    ax = axes[0, component]
    sns.violinplot(data=posterior["mu_i"][:, :, component], fill=False, ax=ax)
    ax.scatter(
        range(N_trajectories),
        synthetic_data["mu_i"][0, :, component],
        color="black",
        marker="x",
        s=80,
        zorder=10,
        label=r"true μi",
    )
    ax.set_title(rf"μi component {component}")

    ax = axes[1, component]
    sns.violinplot(data=posterior["mu_0_i"][:, :, component], fill=False, ax=ax)
    ax.scatter(
        range(N_trajectories),
        synthetic_data["mu_0_i"][0, :, component],
        color="black",
        marker="o",
        s=55,
        zorder=10,
        label=r"true μ0,i",
    )
    ax.set_title(rf"μ0,i component {component}")
    ax.set_xlabel("trajectory")
axes[0, 0].set_ylabel("OU mean")
axes[1, 0].set_ylabel("initial mean")
axes[0, 0].legend()
axes[1, 0].legend()
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for component in range(2):
    ax = axes[0, component]
    sns.histplot(posterior["mu_global"][:, component], bins=20, ax=ax)
    ax.axvline(
        true_params["mu_global"][component],
        color="black",
        linestyle="--",
        lw=3,
        label=r"true μglobal",
    )
    ax.set_title(rf"μglobal component {component}")
    ax.legend()

    ax = axes[1, component]
    sns.histplot(posterior["mu_0_global"][:, component], bins=20, ax=ax)
    ax.axvline(
        true_params["mu_0_global"][component],
        color="black",
        linestyle="--",
        lw=3,
        label=r"true μ0,global",
    )
    ax.set_title(rf"μ0,global component {component}")
    ax.legend()
sns.despine()
plt.tight_layout()
plt.show()